# Position Coalescing: Data Exploration

Goal: understand the input data before designing the matching rules. This notebook is exploration only.


## 1. Setup

Find the project folder (the one that contains `assignment/`) so no path is hard coded.


In [1]:
import pandas as pd
from pathlib import Path
from IPython.display import display

current_dir = Path.cwd()
PROJECT_ROOT = next(
    (folder for folder in [current_dir, *current_dir.parents] if (folder / 'assignment').is_dir()),
    None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('No assignment folder found. Open the notebook inside the project.')

ASSIGNMENT_DIR = PROJECT_ROOT / 'assignment'
print('Project folder:', PROJECT_ROOT)


Project folder: /home/Student/revelio_task/position_coalescing


## 2. List the input files

The data is split across many Parquet files. This only lists them, nothing is loaded yet.


In [2]:
import pyarrow.parquet as pq

parquet_files = sorted(ASSIGNMENT_DIR.rglob('*.parquet'))
if not parquet_files:
    raise FileNotFoundError(f'No Parquet files found in {ASSIGNMENT_DIR}')

total_rows = 0
total_size = 0
for file in parquet_files:
    total_rows += pq.ParquetFile(file).metadata.num_rows
    total_size += file.stat().st_size

column_names = pq.ParquetFile(parquet_files[0]).schema.names

print('Files:', len(parquet_files))
print('Rows:', f'{total_rows:,}')
print('Size GB:', round(total_size / 1024 ** 3, 2))
print('Columns:', len(column_names))
print('Column names:', column_names)

Files: 32
Rows: 11,480,179
Size GB: 2.37
Columns: 10
Column names: ['observed_ts', 'user_id', 'company_raw', 'company_website', 'title_raw', 'location_raw', 'description', 'startdate', 'enddate', 'order_in_profile']


## 3. Load a 5% sample of users

Each user has rows spread across many files, so sampling by file gives a biased picture. Instead we read one file at a time and keep every row of users where `user_id % 20 == 0`. Every kept user has a complete history.


In [3]:
dataframes = []
for file in parquet_files:
    part = pd.read_parquet(file)
    dataframes.append(part[part['user_id'] % 20 == 0])
    del part
df = pd.concat(dataframes, ignore_index=True)
del dataframes

print('Shape:', df.shape)
print('Memory GB:', round(df.memory_usage(deep=True).sum() / 1024 ** 3, 2))

Shape: (552761, 10)
Memory GB: 0.33


## 4. Columns and first rows

A first look at the real column names, data types and a few example rows.


In [4]:
print('Columns:', df.columns.tolist())
print()
print(df.dtypes)
display(df.head(5))


Columns: ['observed_ts', 'user_id', 'company_raw', 'company_website', 'title_raw', 'location_raw', 'description', 'startdate', 'enddate', 'order_in_profile']

observed_ts         object
user_id              int64
company_raw         object
company_website     object
title_raw           object
location_raw        object
description         object
startdate           object
enddate             object
order_in_profile     int32
dtype: object


,observed_ts,user_id,company_raw,company_website,title_raw,location_raw,description,startdate,enddate,order_in_profile
0,2021-10-05,565022420,Saint-Gobain,None,Extrusion Specialist,"Niagara Falls, New York, United States",None,2020-02-01,None,1
1,2021-10-05,565022420,Saint Gobain Boron Nitride,None,Technician,"Amherst, NY",None,2017-04-01,2017-05-01,2
2,2021-10-05,565022420,DuPont,None,Chemical worker,United States,Working in production of Metallic Sodium and L...,1987-03-01,2016-12-01,3
3,2021-10-05,565022420,The Chemours Company,None,Process Operator,"Buffalo/Niagara, New York Area",None,1987-03-01,2016-12-01,4
4,2021-10-05,565022420,Saint Gobain Structural Ceramics,None,Forming Operator,"Niagara Falls, New York","Working with 12"" and 24"" iso-press machines @ ...",2018-04-01,None,5


## 5. Compare columns with the assignment

The assignment lists `company_linkedin_url`, but the data has `company_website`. We check what that column really holds.


In [5]:
expected_columns = [
    'observed_ts', 'user_id', 'company_raw', 'company_linkedin_url', 'title_raw',
    'location_raw', 'description', 'startdate', 'enddate', 'order_in_profile'
]
print('Missing from data:', [c for c in expected_columns if c not in df.columns])
print('Extra in data:', [c for c in df.columns if c not in expected_columns])

links = df['company_website'].dropna()
print(links.sample(5, random_state=1).tolist())
print('Share containing linkedin:', round(links.str.contains('linkedin', case=False).mean(), 3))


Missing from data: ['company_linkedin_url']
Extra in data: ['company_website']
['linkedin.com/company/general-atomics', 'linkedin.com/company/intercon', 'linkedin.com/company/techstars', 'linkedin.com/company/stephenson-mk-trust', 'linkedin.com/company/university-of-massachusetts-lowell']
Share containing linkedin: 1.0


## 6. Missing values

Share of rows with an empty value in each column. Empty does not mean bad. A real job can lack a description or end date.


In [6]:
missing_fraction = df.isna().mean().sort_values(ascending=False)
display(missing_fraction.rename('fraction_missing').to_frame())


,fraction_missing
description,0.638746
location_raw,0.468557
company_website,0.445243
enddate,0.412095
startdate,0.159045
title_raw,0.009509
company_raw,0.003555
observed_ts,0.000000
user_id,0.000000
order_in_profile,0.000000


## 7. Convert dates

Dates load as plain Python objects. We convert them to real dates. Values pandas cannot store become `NaT` (missing), so we count how many were lost this way.


In [7]:
date_columns = ['observed_ts', 'startdate', 'enddate']

for column in date_columns:
    missing_before = df[column].isna().sum()
    df[column] = pd.to_datetime(df[column], errors='coerce')
    print(column, 'turned into NaT:', int(df[column].isna().sum() - missing_before))


observed_ts turned into NaT: 0
startdate turned into NaT: 2
enddate turned into NaT: 1


## 8. Date ranges and suspicious dates

Counts only. These are not cleaning rules yet.


In [8]:
for column in date_columns:
    print(column, ':', df[column].min(), 'to', df[column].max())

end_before_start = df['startdate'].notna() & df['enddate'].notna() & (df['enddate'] < df['startdate'])
start_after_snapshot = df['startdate'] > df['observed_ts']
end_after_snapshot = df['enddate'] > df['observed_ts']
start_before_1950 = df['startdate'] < pd.Timestamp('1950-01-01')
start_is_1900 = df['startdate'] == pd.Timestamp('1900-01-01')

print()
print('End before start:', int(end_before_start.sum()))
print('Start after its snapshot date:', int(start_after_snapshot.sum()))
print('End after its snapshot date:', int(end_after_snapshot.sum()))
print('Start before 1950:', int(start_before_1950.sum()))
print('Start exactly 1900 01 01:', int(start_is_1900.sum()))


observed_ts : 2016-09-30 00:00:00 to 2026-10-05 00:00:00
startdate : 1900-01-01 00:00:00 to 2026-09-01 00:00:00
enddate : 1921-01-01 00:00:00 to 2096-06-01 00:00:00

End before start: 88
Start after its snapshot date: 3
End after its snapshot date: 67
Start before 1950: 66
Start exactly 1900 01 01: 37


## 9. Snapshots per user

A snapshot is one saved copy of a profile on one date. Here we count how many snapshots each user has.


In [9]:
observations_per_user = df.groupby('user_id')['observed_ts'].nunique()

print(observations_per_user.describe())
print('Share of users with one snapshot:', round((observations_per_user == 1).mean(), 3))


count    7531.000000
mean       26.372859
std        18.172111
min         1.000000
25%         9.000000
50%        27.000000
75%        38.000000
max       131.000000
Name: observed_ts, dtype: float64
Share of users with one snapshot: 0.029


## 10. Do snapshot sizes change?

Share of users (with 2 or more snapshots) whose number of job rows differs between snapshots.


In [10]:
snapshot_sizes = df.groupby(['user_id', 'observed_ts']).size().rename('row_count').reset_index()
multi_users = observations_per_user[observations_per_user >= 2].index
sizes_per_user = snapshot_sizes[snapshot_sizes['user_id'].isin(multi_users)].groupby('user_id')['row_count'].nunique()

print('Users with 2 or more snapshots:', len(multi_users))
print('Share whose row count changes:', round((sizes_per_user > 1).mean(), 3))


Users with 2 or more snapshots: 7310
Share whose row count changes: 0.635


## 11. Thin rows and thin snapshots

A thin snapshot is one where every job row has very little filled in (for example only a company name). We count filled fields per row, then find snapshots where no row has more than 1 or 2 filled fields.


In [11]:
useful_columns = ['company_raw', 'title_raw', 'location_raw', 'description', 'startdate', 'enddate']
df['n_filled'] = df[useful_columns].notna().sum(axis=1)

display(df['n_filled'].value_counts().sort_index().rename('row_count').to_frame())

max_filled_per_snapshot = df.groupby(['user_id', 'observed_ts'])['n_filled'].max()
print('Total snapshots:', len(max_filled_per_snapshot))
print('Thin, at most 1 field:', int((max_filled_per_snapshot <= 1).sum()))
print('Thin, at most 2 fields:', int((max_filled_per_snapshot <= 2).sum()))


,row_count
n_filled,
0,78
1,3961
2,82885
3,49302
4,133332
5,168619
6,114584


Total snapshots: 198614
Thin, at most 1 field: 3727
Thin, at most 2 fields: 75790


## 12. Do users span several files?

Counts how many users appear in 1 file, 2 files, and so on. This decides how work must be split for scale.


In [12]:
pairs = pd.concat([
    pd.read_parquet(file, columns=['user_id']).drop_duplicates().assign(file=file.name)
    for file in parquet_files
])
print(pairs.groupby('user_id')['file'].nunique().value_counts().sort_index())


file
1     4645
2     7093
3     5820
4     5034
5     5207
6     4671
7     3714
8     3152
9     2926
10    2881
11    2893
12    3051
13    3403
14    3638
15    3995
16    4675
17    5617
18    6577
19    7881
20    8605
21    8853
22    8591
23    7643
24    6820
25    6162
26    5241
27    4320
28    3048
29    1921
30    1106
31     561
32     256
Name: count, dtype: int64


## 13. How often does a job survive to the next snapshot?

For each job row we ask: does the same job appear in the user's next snapshot? We test five ways of defining the same job. A loose key scores higher but can mix different jobs, so a high score alone is not proof of a good key.


In [13]:
d = df[['user_id', 'observed_ts', 'company_raw', 'title_raw', 'startdate', 'company_website']].drop_duplicates()
ts = d[['user_id', 'observed_ts']].drop_duplicates().sort_values(['user_id', 'observed_ts'], ascending=False)
ts['next_ts'] = ts.groupby('user_id')['observed_ts'].shift(1)
cur = d.merge(ts, on=['user_id', 'observed_ts']).dropna(subset=['next_ts'])
nxt = d.rename(columns={'observed_ts': 'next_ts'})

key_sets = {
    'company + title + start': ['company_raw', 'title_raw', 'startdate'],
    'company + start': ['company_raw', 'startdate'],
    'company + title': ['company_raw', 'title_raw'],
    'url + start': ['company_website', 'startdate'],
    'company only': ['company_raw'],
}
for name, key in key_sets.items():
    key = ['user_id'] + key
    right = nxt[key + ['next_ts']].drop_duplicates().assign(found=1)
    hit_rate = cur.merge(right, on=key + ['next_ts'], how='left')['found'].notna().mean()
    print(name, round(hit_rate, 3))


company + title + start 0.924
company + start 0.945
company + title 0.93
url + start 0.821
company only 0.963


## 14. What do unmatched jobs look like?

Jobs not found in the next snapshot using company + title + start. For 5 random cases we show the job before and the full next snapshot.


In [14]:
d = df[['user_id', 'observed_ts', 'company_raw', 'title_raw', 'startdate', 'enddate']].drop_duplicates()
ts = d[['user_id', 'observed_ts']].drop_duplicates().sort_values(['user_id', 'observed_ts'], ascending=False)
ts['next_ts'] = ts.groupby('user_id')['observed_ts'].shift(1)
cur = d.merge(ts, on=['user_id', 'observed_ts']).dropna(subset=['next_ts'])
key = ['user_id', 'company_raw', 'title_raw', 'startdate']
right = d.rename(columns={'observed_ts': 'next_ts'})[key + ['next_ts']].drop_duplicates().assign(found=1)
matched = cur.merge(right, on=key + ['next_ts'], how='left')
lost = matched[matched['found'].isna()]
print('Unmatched rows:', len(lost), 'of', len(matched))

for _, row in lost.sample(5, random_state=2).iterrows():
    print('\nUser', row['user_id'], '| between', row['observed_ts'].date(), 'and', row['next_ts'].date())
    print('BEFORE:', row['company_raw'], '|', row['title_raw'], '|', row['startdate'], '|', row['enddate'])
    next_rows = d[(d['user_id'] == row['user_id']) & (d['observed_ts'] == row['next_ts'])]
    print(next_rows[['company_raw', 'title_raw', 'startdate', 'enddate']].head(8).to_string(index=False))


Unmatched rows: 39779 of 521851

User 328736160 | between 2024-06-30 and 2024-07-10
BEFORE: Biogen | Engineer II, Manufacturing Sciences | 2012-07-01 00:00:00 | 2013-12-01 00:00:00
                                                  company_raw                           title_raw  startdate    enddate
                                             PAK BioSolutions                     CEO and Founder 2019-08-01        NaT
                                                    MedImmune                         Scientist I 2017-06-01 2018-06-01
                                                    MedImmune              Associate Scientist II 2014-03-01 2017-06-01
                                                      Novavax                       Tech Transfer 2013-12-01 2014-03-01
                                                  Biogen Idec Engineer II, Manufacturing Sciences 2012-07-01 2013-12-01
Tissue and Metabolic Engineering Laboratory, Tufts University         Graduate Research Assistant 2

## 15. Second sample check

Repeat a few key numbers on a different 5% of users (`user_id % 20 == 7`). If they match the first sample, the sample is a reliable base for design.


In [15]:
parts = []
for file in parquet_files:
    part = pd.read_parquet(file)
    parts.append(part[part['user_id'] % 20 == 7])
    del part
sample_b = pd.concat(parts, ignore_index=True)
del parts

obs_b = sample_b.groupby('user_id')['observed_ts'].nunique()
sizes_b = sample_b.groupby(['user_id', 'observed_ts']).size().rename('row_count').reset_index()
changes_b = sizes_b[sizes_b['user_id'].isin(obs_b[obs_b >= 2].index)].groupby('user_id')['row_count'].nunique()

print('Rows:', len(sample_b), 'Users:', sample_b['user_id'].nunique())
print('Median snapshots per user:', obs_b.median())
print('Share with one snapshot:', round((obs_b == 1).mean(), 3))
print('Share whose row count changes:', round((changes_b > 1).mean(), 3))


Rows: 563641 Users: 7481
Median snapshots per user: 27.0
Share with one snapshot: 0.029
Share whose row count changes: 0.627


## 16. The 1900 01 01 start date

Earlier, one check found 37 rows with a start date of 1900 01 01 and another found 0. The data was fine. The second check ran before the dates were converted, so it compared a plain Python date with a Timestamp, which is always False.

**Confirmed:** 37 rows, all from one user (a profile called "My Damn Self"), repeated across that user's snapshots. After conversion both checks agree.

**Assumption:** 1900 01 01 is probably a placeholder for an unknown date. It is rare, so no special rule is needed yet.

## 17. Match each job to the next snapshot, tier by tier

**What it does:** for each job row, looks in the same user's next snapshot and tries four keys from strict to loose: company + title + start, company + start, company + title, company only. Each row gets the first tier that finds it, and we count how many candidates fit.

**Why it matters:** it shows how many jobs we can follow, and how risky each key is.

**Confirmed:**
* 77.8% match on all three fields, 14.4% on company + title, 1.9% on company + start, 1.6% on company only. 4.3% find no match.
* The strict tiers almost always have exactly one candidate (98% or more). Company only is weak: 14% of its matches have 2 or more candidates.
* A missing field never counts as a match.

Later sections reuse `cur`, `nxt`, `snaps` and `tiers` from this cell, so run it before sections 18 to 24.

In [16]:
d = df[['user_id', 'observed_ts', 'company_raw', 'title_raw', 'startdate', 'enddate', 'location_raw']].copy()
d['row_id'] = range(len(d))

snaps = d[['user_id', 'observed_ts']].drop_duplicates().sort_values(['user_id', 'observed_ts'])
snaps['next_ts'] = snaps.groupby('user_id')['observed_ts'].shift(-1)
snaps = snaps.dropna(subset=['next_ts'])

cur = d.merge(snaps, on=['user_id', 'observed_ts'])
nxt = d.rename(columns={'observed_ts': 'next_ts'})

tiers = {
    'company + title + start': ['company_raw', 'title_raw', 'startdate'],
    'company + start': ['company_raw', 'startdate'],
    'company + title': ['company_raw', 'title_raw'],
    'company only': ['company_raw'],
}

def count_candidates(keys):
    left = cur.loc[cur[keys].notna().all(axis=1), ['row_id', 'user_id', 'next_ts'] + keys]
    right = nxt.loc[nxt[keys].notna().all(axis=1), ['row_id', 'user_id', 'next_ts'] + keys]
    merged = left.merge(right, on=['user_id', 'next_ts'] + keys, suffixes=('', '_next'))
    return merged.groupby('row_id')['row_id_next'].nunique()

cur['tier'] = 'no match'
cur['n_candidates'] = 0
for name, keys in tiers.items():
    counts = count_candidates(keys)
    todo = (cur['tier'] == 'no match') & cur['row_id'].isin(counts.index)
    cur.loc[todo, 'tier'] = name
    cur.loc[todo, 'n_candidates'] = cur.loc[todo, 'row_id'].map(counts)

print('Job rows compared:', len(cur))
print(cur['tier'].value_counts(normalize=True).round(3))
print()
print('Share by number of candidates (3 means 3 or more):')
print(pd.crosstab(cur['tier'], cur['n_candidates'].clip(upper=3), normalize='index').round(3))

Job rows compared: 530366
tier
company + title + start    0.778
company + title            0.144
no match                   0.043
company + start            0.019
company only               0.016
Name: proportion, dtype: float64

Share by number of candidates (3 means 3 or more):
n_candidates               0      1      2      3
tier                                             
company + start          0.0  0.977  0.014  0.008
company + title          0.0  0.978  0.012  0.010
company + title + start  0.0  0.987  0.009  0.005
company only             0.0  0.862  0.074  0.064
no match                 1.0  0.000  0.000  0.000


## 18. What changes inside matched jobs

**What it does:** for jobs matched to exactly one candidate in section 17, compares title, start date, end date and location between the two snapshots. Each field is `same`, `filled` (was blank), `blanked` (became blank) or `changed`.

**Confirmed:** 96.8% of titles, 98.4% of start dates and 98.0% of end dates stay the same. Location is the noisiest (93.2% same). Fields get filled and blanked at similar rates.

**Why it matters:** a job with a small edit is still the same job. An edit must update the position, not create a new one.

Start date shows no change in two tiers because start date is part of their key.

In [17]:
pairs = []
for name, keys in tiers.items():
    left = cur.loc[(cur['tier'] == name) & (cur['n_candidates'] == 1), ['row_id', 'user_id', 'next_ts'] + keys]
    right = nxt[['row_id', 'user_id', 'next_ts'] + keys]
    merged = left.merge(right, on=['user_id', 'next_ts'] + keys, suffixes=('', '_next'))
    merged = merged[['row_id', 'row_id_next']].assign(tier=name)
    pairs.append(merged)
pairs = pd.concat(pairs).reset_index(drop=True)

fields = ['title_raw', 'startdate', 'enddate', 'location_raw']
lookup = d.set_index('row_id')[fields]
old = lookup.loc[pairs['row_id']].reset_index(drop=True).add_suffix('_old')
new = lookup.loc[pairs['row_id_next']].reset_index(drop=True).add_suffix('_new')
cmp = pd.concat([pairs, old, new], axis=1)

def change_type(field):
    o, n = cmp[field + '_old'], cmp[field + '_new']
    out = pd.Series('same', index=cmp.index)
    out[o.isna() & n.notna()] = 'filled'
    out[o.notna() & n.isna()] = 'blanked'
    out[o.notna() & n.notna() & (o != n)] = 'changed'
    return out

for field in fields:
    cmp[field] = change_type(field)

print('Matched pairs:', len(cmp))
print(pd.DataFrame({f: cmp[f].value_counts(normalize=True) for f in fields}).round(3))

print()
print('Start date change type by tier:')
print(pd.crosstab(cmp['tier'], cmp['startdate'], normalize='index').round(3))

ex = cmp[(cmp['tier'] == 'company + title') & (cmp['startdate'] == 'changed')]
display(ex.sample(min(5, len(ex)), random_state=1)[['startdate_old', 'startdate_new', 'enddate_old', 'enddate_new']])

ex = cmp[(cmp['tier'] == 'company + start') & (cmp['title_raw'] == 'changed')]
display(ex.sample(min(5, len(ex)), random_state=1)[['title_raw_old', 'title_raw_new']])

Matched pairs: 499088
         title_raw  startdate  enddate  location_raw
blanked      0.008      0.006    0.006         0.025
changed      0.017      0.006    0.004         0.015
filled       0.006      0.005    0.010         0.028
same         0.968      0.984    0.980         0.932

Start date change type by tier:
startdate                blanked  changed  filled   same
tier                                                    
company + start            0.000    0.000   0.000  1.000
company + title            0.013    0.016   0.017  0.954
company + title + start    0.000    0.000   0.000  1.000
company only               0.274    0.230   0.150  0.346


,startdate_old,startdate_new,enddate_old,enddate_new
481564,2022-08-01,2022-09-01,NaT,NaT
431974,2021-03-01,2021-01-01,2022-01-01,2022-01-01
462723,2014-09-01,2014-01-01,2015-09-01,2015-01-01
486290,2015-10-01,2012-11-01,2016-03-01,2014-08-01
480285,2014-08-01,2008-06-01,NaT,2014-08-01


,title_raw_old,title_raw_new
415084,x ray technologist,radiologic technologist/ cardiac cath lab
410475,Senior Client Development Consultant,Senior Client Development Consultant
412667,Football Scout,Football Scout
415804,Building Surveying Graduate,Building Surveying Graduate
413283,Recepcionista,Receptionist


## 19. Title and start date changes

**What it does:** for pairs whose title changed, checks if it is only spaces, case or unicode, if one title extends the other, or if the wording really differs. For pairs whose start date changed, measures how far it moved.

**Confirmed:**
* Of 8,709 changed titles, 44.5% are the same after cleanup, 5.6% are an extension, 49.9% are different wording.
* Of 1,156 changed start dates, 57.4% moved by less than 12 months and 47.4% stayed in the same year.
* January is common among changed start dates (about 20% of old and 17% of new, against 8% by chance).

**Assumption:** January is probably a placeholder month. Not tested.

**Why it matters:** text cleaning removes nearly half of title changes. Start dates are mostly stable but can shift a little, so a start date difference alone should not prove two different jobs.

In [18]:
import unicodedata

def normalize(text):
    text = unicodedata.normalize('NFKC', str(text)).lower()
    return ' '.join(text.split())

t = cmp[cmp['title_raw'] == 'changed'].copy()
old_t = t['title_raw_old'].map(normalize)
new_t = t['title_raw_new'].map(normalize)
t['cosmetic'] = old_t == new_t
t['extension'] = ~t['cosmetic'] & pd.Series([n.startswith(o) or o.startswith(n) for o, n in zip(old_t, new_t)], index=t.index)

print('Title changed pairs:', len(t))
print('Same after cleanup:', round(t['cosmetic'].mean(), 3))
print('One title extends the other:', round(t['extension'].mean(), 3))
print('Different wording:', round((~t['cosmetic'] & ~t['extension']).mean(), 3))
cosmetic = t[t['cosmetic']]
for _, r in cosmetic.sample(min(3, len(cosmetic)), random_state=1).iterrows():
    print(repr(r['title_raw_old']), repr(r['title_raw_new']))

s = cmp[(cmp['tier'] == 'company + title') & (cmp['startdate'] == 'changed')].copy()
old_s = pd.to_datetime(s['startdate_old'], errors='coerce')
new_s = pd.to_datetime(s['startdate_new'], errors='coerce')
valid = old_s.notna() & new_s.notna()
print('Dates pandas cannot store:', int((~valid).sum()))
s, old_s, new_s = s[valid], old_s[valid], new_s[valid]
gap = ((new_s.dt.year - old_s.dt.year) * 12 + (new_s.dt.month - old_s.dt.month)).abs()

print()
print('Start changed pairs:', len(s))
print('Same year:', round((old_s.dt.year == new_s.dt.year).mean(), 3))
print('Old start in January:', round((old_s.dt.month == 1).mean(), 3))
print('New start in January:', round((new_s.dt.month == 1).mean(), 3))
print(pd.cut(gap, [-1, 11, 24, 120, 10000], labels=['under 12 months', '12 to 24', '25 to 120', 'over 120']).value_counts(normalize=True).round(3))

Title changed pairs: 8709
Same after cleanup: 0.445
One title extends the other: 0.056
Different wording: 0.499
'CEO/MD' 'CEO/MD '
'Auxuliar de control de acceso ' 'Auxuliar de control de acceso'
'Customer Service Technical Agent' 'Customer Service Technical Agent\xa0'
Dates pandas cannot store: 0

Start changed pairs: 1156
Same year: 0.474
Old start in January: 0.198
New start in January: 0.174
under 12 months    0.574
25 to 120          0.189
12 to 24           0.188
over 120           0.049
Name: proportion, dtype: float64


## 20. Why do some jobs find no match?

**What it does:** takes the 22,799 jobs with no match in section 17 and checks four things. Is the company missing? Does the same company come back in a later snapshot? Does the row have a LinkedIn URL? Does the same URL appear in the next snapshot under a new company name?

**Confirmed:** 7.7% have no company, 59.3% have a company that returns later, 46.3% have a URL, and 9.4% show the same URL under a new name (for example `KKR & Co. Inc.` became `KKR`).

**Assumptions:** these groups overlap, so do not add them up. "Returns later" could be a rehire, not the same job.

**Why it matters:** we carry deleted jobs forward, so a deletion and a temporary gap give the same output. The costly mistake is a missed rename, because it creates a second position for one job.

In [19]:
nm = cur.loc[cur['tier'] == 'no match', ['row_id', 'user_id', 'observed_ts', 'next_ts', 'company_raw']].copy()
nm['observed_ts'] = pd.to_datetime(nm['observed_ts'])
nm['next_ts'] = pd.to_datetime(nm['next_ts'])

e = df[['user_id', 'observed_ts', 'company_raw', 'company_website']].drop_duplicates()

named = nm[nm['company_raw'].notna()]
later = named.merge(
    e[['user_id', 'observed_ts', 'company_raw']].rename(columns={'observed_ts': 'later_ts'}),
    on=['user_id', 'company_raw']
)
later = later[later['later_ts'] > later['next_ts']]
returns = named['row_id'].isin(later['row_id'])

urls = nm.merge(
    e.dropna(subset=['company_website']),
    on=['user_id', 'observed_ts', 'company_raw']
).drop_duplicates('row_id')

next_urls = (
    e.dropna(subset=['company_website'])
    .rename(columns={'observed_ts': 'next_ts', 'company_raw': 'new_company'})
    .drop_duplicates()
)
renamed = urls.merge(next_urls, on=['user_id', 'next_ts', 'company_website']).drop_duplicates('row_id')

print('No match rows:', len(nm))
print('Company missing in the row:', round(nm['company_raw'].isna().mean(), 3))
print('Same company returns in a later snapshot:', round(returns.sum() / len(nm), 3))
print('Row has a LinkedIn URL:', round(len(urls) / len(nm), 3))
print('Same URL in next snapshot under a new name:', round(len(renamed) / len(nm), 3))
display(renamed.sample(min(5, len(renamed)), random_state=1)[['company_raw', 'new_company']])

No match rows: 22799
Company missing in the row: 0.077
Same company returns in a later snapshot: 0.593
Row has a LinkedIn URL: 0.463
Same URL in next snapshot under a new name: 0.094


,company_raw,new_company
1974,"Honeywell International, Inc. (formerly Allied...",Honeywell
2178,Paradigma Business Solutions S.A.,Paradigma Business Solutions Ltda
377,"Ktedra Innovation, Learning and Technology",Ktedra Technology and Innovation
1424,Amazon.com,Amazon
1296,Goglio S.p.A.,Goglio Group


## 21. One reason per no match row, and the hard cases

**What it does:** part A gives each of the 22,799 rows a single reason, in priority order. Part B counts risky situations inside one snapshot: duplicate rows, companies with several jobs, and the same company and title with different start dates.

**Confirmed:**
* Reasons: company returns later 53.4%, unexplained 29.7%, renamed with same URL 9.2%, company missing 7.7%.
* 3.5% of rows are exact duplicates of another row in the same snapshot (2.9% counting only rows with company, title and start). They sit in 2.4% of snapshots.
* 20.9% of jobs are in a company where the person has 2 or more jobs.
* 2,703 groups have the same company and title with different start dates. The 3 examples we read are real separate stints (two internships, four short jobs, a rehire).

**Why it matters:** start date must separate stints, matching must be one to one, and identical rows should collapse into one position. Matching on company only is risky.

Only 3 examples were read, so "different start date means different job" is likely, not proven.

In [20]:
nm['reason'] = 'unexplained'
nm.loc[nm['row_id'].isin(later['row_id']), 'reason'] = 'company returns later'
nm.loc[nm['row_id'].isin(renamed['row_id']), 'reason'] = 'renamed, same URL'
nm.loc[nm['company_raw'].isna(), 'reason'] = 'company missing'
print(nm['reason'].value_counts(normalize=True).round(3))

jobs = df[['user_id', 'observed_ts', 'company_raw', 'title_raw', 'startdate']]
total_snapshots = jobs[['user_id', 'observed_ts']].drop_duplicates().shape[0]

dup = jobs.duplicated(keep=False)
complete = jobs.notna().all(axis=1)
dup_complete = jobs[complete].duplicated(keep=False)
dup_snapshots = jobs[dup][['user_id', 'observed_ts']].drop_duplicates().shape[0]

print()
print('Rows identical to another row in the same snapshot:', round(dup.mean(), 3))
print('Same, counting only rows with company, title and start:', round(dup_complete.mean(), 3))
print('Snapshots containing such duplicates:', round(dup_snapshots / total_snapshots, 3))

distinct = jobs.dropna(subset=['company_raw']).drop_duplicates()
per_company = distinct.groupby(['user_id', 'observed_ts', 'company_raw']).size()
print('Distinct job rows in a company with 2 or more jobs:', round(per_company[per_company > 1].sum() / per_company.sum(), 3))

per_title = (
    distinct.dropna(subset=['title_raw'])
    .groupby(['user_id', 'observed_ts', 'company_raw', 'title_raw'])['startdate'].nunique()
)
multi_start = per_title[per_title > 1]
print('Same company and title with 2 or more start dates (groups):', len(multi_start))

keys = multi_start.reset_index().sample(min(3, len(multi_start)), random_state=1)
for _, k in keys.iterrows():
    mask = (
        (df['user_id'] == k['user_id']) & (df['observed_ts'] == k['observed_ts'])
        & (df['company_raw'] == k['company_raw']) & (df['title_raw'] == k['title_raw'])
    )
    print()
    print(df.loc[mask, ['company_raw', 'title_raw', 'startdate', 'enddate']].to_string(index=False))

reason
company returns later    0.534
unexplained              0.297
renamed, same URL        0.092
company missing          0.077
Name: proportion, dtype: float64

Rows identical to another row in the same snapshot: 0.035
Same, counting only rows with company, title and start: 0.029
Snapshots containing such duplicates: 0.024
Distinct job rows in a company with 2 or more jobs: 0.209
Same company and title with 2 or more start dates (groups): 2703

    company_raw title_raw  startdate    enddate
Algérie Télécom Stagiaire 2022-02-01 2022-05-01
Algérie Télécom Stagiaire 2019-11-01 2019-12-01

                     company_raw               title_raw  startdate    enddate
MAISON DE QUARTIER DES PYRAMIDES Moniteur de centre aéré 2018-10-01 2018-10-01
MAISON DE QUARTIER DES PYRAMIDES Moniteur de centre aéré 2018-03-01 2018-03-01
MAISON DE QUARTIER DES PYRAMIDES Moniteur de centre aéré 2017-12-01 2018-01-01
MAISON DE QUARTIER DES PYRAMIDES Moniteur de centre aéré 2017-04-01 2017-04-01

compan

## 22. Jobs that appear in the next snapshot

**What it does:** the reverse direction. For every row in a next snapshot, checks if the same job existed in the previous one (same four tiers). Rows that did not are split by what their start date says.

**Confirmed:**
* 94.8% of the 537,909 rows were already in the previous snapshot.
* 27,724 rows look new. Of these, 73.5% start before the previous snapshot, 15.3% have no start date, 6.6% have no company, and only 4.5% (about 1,250 rows) start after it. Only that last group is clearly a new job.

**Assumption:** many of the "new looking" rows are edited versions of old jobs. Section 23 tests this.

**Why it matters:** a row that looks new is usually an old job. Treating them all as new creates duplicate positions.

In [21]:
new = nxt[['row_id', 'user_id', 'next_ts', 'company_raw', 'title_raw', 'startdate']].merge(
    snaps, on=['user_id', 'next_ts'])

found = set()
for keys in tiers.values():
    l = cur.loc[cur[keys].notna().all(axis=1), ['user_id', 'next_ts'] + keys].drop_duplicates()
    r = new.loc[new[keys].notna().all(axis=1), ['row_id', 'user_id', 'next_ts'] + keys]
    found |= set(r.merge(l, on=['user_id', 'next_ts'] + keys)['row_id'])

new['seen_before'] = new['row_id'].isin(found)
print('Rows in next snapshots:', f'{len(new):,}')
print('Share that appear in the previous snapshot:', round(new['seen_before'].mean(), 3))

nw = new[~new['seen_before']].copy()
start = pd.to_datetime(nw['startdate'], errors='coerce')
prev = pd.to_datetime(nw['observed_ts'])
nw['kind'] = 'start before previous snapshot'
nw.loc[start > prev, 'kind'] = 'start after previous snapshot'
nw.loc[start.isna(), 'kind'] = 'start missing'
nw.loc[nw['company_raw'].isna(), 'kind'] = 'company missing'
print()
print('New looking rows:', f'{len(nw):,}')
print(nw['kind'].value_counts(normalize=True).round(3))

print()
ex = nw[nw['kind'] == 'start before previous snapshot'].sample(5, random_state=1)
print(ex[['company_raw', 'title_raw', 'startdate', 'observed_ts', 'next_ts']].to_string(index=False))

Rows in next snapshots: 537,909
Share that appear in the previous snapshot: 0.948

New looking rows: 27,724
kind
start before previous snapshot    0.735
start missing                     0.153
company missing                   0.066
start after previous snapshot     0.045
Name: proportion, dtype: float64

                     company_raw                                   title_raw  startdate observed_ts    next_ts
Iowa Department of Public Health COVID-19 Case Investigator & Contact Tracer 2020-11-01  2025-09-01 2025-09-21
             Partners HealthCare                  Desktop Support Technician 2013-02-01  2024-05-29 2024-06-05
            Irvin Automotive LLC                           Director of Sales 2016-09-01  2023-09-05 2023-10-09
                   Luth Research                                  Consultant 2019-03-01  2024-08-01 2024-08-11
            The Hotel Collection                             Revenue Manager 2013-08-01  2025-06-15 2025-06-28


## 23. Do unmatched old jobs pair up with new looking jobs?

**What it does:** within one user and two consecutive snapshots, pairs an unmatched old job with a new looking job that has the same start date. A pair is clean when each side has exactly one partner.

**Confirmed:**
* 36.7% of 18,261 old jobs and 31.5% of 22,414 new jobs have a partner. There are 4,047 clean pairs.
* All 6 examples are the same job after an edit: punctuation, case, a longer company name, a dropped legal suffix, a changed title.

**Open:** one example (Equityvest) looked identical but did not match. The likely cause is an invisible character, but we did not check that row directly.

**Why it matters:** start date can recover renames that text matching misses, but only about a third of them.

In [22]:
cols = ['row_id', 'user_id', 'next_ts', 'company_raw', 'title_raw', 'startdate']
old = cur.loc[cur['tier'] == 'no match', cols].dropna(subset=['startdate'])
newu = nw[cols].dropna(subset=['startdate'])

pr = old.merge(newu, on=['user_id', 'next_ts', 'startdate'], suffixes=('_old', '_new'))

print('Unmatched old jobs with a start date:', f'{len(old):,}')
print('New looking jobs with a start date:', f'{len(newu):,}')
print('Old jobs with at least one same start date partner:', round(pr['row_id_old'].nunique() / len(old), 3))
print('New jobs with at least one same start date partner:', round(pr['row_id_new'].nunique() / len(newu), 3))

old_n = pr.groupby('row_id_old')['row_id_new'].nunique()
new_n = pr.groupby('row_id_new')['row_id_old'].nunique()
pr['clean'] = pr['row_id_old'].map(old_n).eq(1) & pr['row_id_new'].map(new_n).eq(1)
print('Clean one to one pairs:', f'{pr["clean"].sum():,}')

print()
show = pr[pr['clean']].sample(6, random_state=1)
print(show[['company_raw_old', 'company_raw_new', 'title_raw_old', 'title_raw_new', 'startdate']].to_string(index=False))

Unmatched old jobs with a start date: 18,261
New looking jobs with a start date: 22,414
Old jobs with at least one same start date partner: 0.367
New jobs with at least one same start date partner: 0.315
Clean one to one pairs: 4,047

            company_raw_old             company_raw_new                           title_raw_old                           title_raw_new  startdate
 Pt industri nabati lestari Pt industri nabati lestari                   DCS - Foreman Refinery                 DCS - Foreman Refinery  2023-03-01
               PolyPid Ltd.               PolyPid Ltd.  EVP Regulatory Affairs, Clinical & R& D EVP Regulatory Affairs, Clinical & R& D 2022-03-01
ACADIA Pharmaceuticals Inc.      acadia pharmaceuticals  Executive Neuroscience Specialty Sales  Executive Neuroscience Specialty Sales 2016-04-01
Carewatch Care Services Ltd     carewatch care services                          Care Assistant                          care assistant 2010-01-01
 LDF Sales and Distributing LD

## 24. What changed inside the clean pairs

**What it does:** splits the 4,047 clean pairs by what differs after text cleaning, then prints the riskiest group (company and title both differ).

**Confirmed:**
* Company changed only 42.0%, same after cleaning 30.3% (case, trailing space), a field is missing 18.3%, both differ 9.0%, title changed only 0.5%.
* Of 8 risky examples, 7 look like the same job (garbled text, translation, a longer or new company name). 1 is doubtful: `Orange and Vision TV` became `The Walt Disney Company`.

**Caution:** the 0.5% for title only is a side effect of how the pairs were picked. Such pairs would already have matched on company + start in section 17.

**Why it matters:** about a third of these pairs only need text cleaning. The rest need start date plus a guard against wrong merges.

In [23]:
import numpy as np

cp = pr[pr['clean']].copy()
norm = lambda x: normalize(x) if isinstance(x, str) else None
cn_old, cn_new = cp['company_raw_old'].map(norm), cp['company_raw_new'].map(norm)
tn_old, tn_new = cp['title_raw_old'].map(norm), cp['title_raw_new'].map(norm)

missing = cn_old.isna() | cn_new.isna() | tn_old.isna() | tn_new.isna()
cs = cn_old.eq(cn_new) & ~missing
ts = tn_old.eq(tn_new) & ~missing

cp['what'] = 'both differ'
cp.loc[cs & ~ts, 'what'] = 'title changed only'
cp.loc[~cs & ts, 'what'] = 'company changed only'
cp.loc[cs & ts, 'what'] = 'same after cleaning'
cp.loc[missing, 'what'] = 'a field is missing'
print(cp['what'].value_counts(normalize=True).round(3))

print()
same = cp[cp['what'] == 'same after cleaning'].head(3)
for _, r in same.iterrows():
    print(repr(r['company_raw_old']), repr(r['company_raw_new']))
    print(repr(r['title_raw_old']), repr(r['title_raw_new']))

print()
risky = cp[cp['what'] == 'both differ'].sample(8, random_state=1)
print(risky[['company_raw_old', 'company_raw_new', 'title_raw_old', 'title_raw_new', 'startdate']].to_string(index=False))

what
company changed only    0.420
same after cleaning     0.303
a field is missing      0.183
both differ             0.090
title changed only      0.005
Name: proportion, dtype: float64

'Child Language and Speech Studies (CLASS) Lab' 'Child Language and Speech Studies (CLASS) Lab '
'Research Assistant' 'Research Assistant'
'Energy Management Specialists' 'Energy Management Specialists '
'Heating Air Conditioning Technician' 'Heating Air Conditioning Technician'
'KIPA LATAM' 'KIPA\xa0LATAM'
'Consultoría en Tecnología, formación de talento y datos' 'Consultoría en Tecnología, formación de talento y datos'

                                                                              company_raw_old                                                                     company_raw_new                            title_raw_old                                       title_raw_new  startdate
                                                                          Profesor Particular          

## 25. Garbled text

**What it does:** finds values broken by an encoding error (UTF 8 text read as Windows text) that can be repaired, and checks whether whole snapshots are affected.

**Confirmed:** 22 distinct garbled values, about 0.0% of rows. Only 14 snapshots (0.01%) have any, and 3 of them are 90% or more garbled. Repairs work: `AdÄ±yaman UÌˆniversitesi` becomes `Adıyaman Üniversitesi`.

**Assumptions:** the detector is strict, so 22 is a lower bound. We do not know if these 3 snapshots are the "corrupted snapshots" of the assignment.

**Why it matters:** low priority. A small repair step can come later.

In [24]:
def fix(s):
    for enc in ('cp1252', 'latin1'):
        try:
            return s.encode(enc).decode('utf8')
        except (UnicodeEncodeError, UnicodeDecodeError):
            pass
    return None

fixed = {}
for col in ['company_raw', 'title_raw']:
    for v in df[col].dropna().unique():
        f = fix(v)
        if f is not None and f != v:
            fixed[v] = f
print('Distinct garbled values:', f'{len(fixed):,}')

g = df[['user_id', 'observed_ts', 'company_raw', 'title_raw']].copy()
g['garbled'] = g['company_raw'].isin(fixed) | g['title_raw'].isin(fixed)
print('Rows with garbled text:', round(g['garbled'].mean(), 4))

per_snap = g.groupby(['user_id', 'observed_ts'])['garbled'].mean()
bad = per_snap[per_snap > 0]
print('Snapshots with any garbled row:', round(len(bad) / len(per_snap), 4))
print('Snapshots with at least 90% garbled rows:', int((per_snap >= 0.9).sum()))
print(bad.describe().round(3))

print()
for k in list(fixed)[:3]:
    print(repr(k), '->', repr(fixed[k]))

Distinct garbled values: 22
Rows with garbled text: 0.0
Snapshots with any garbled row: 0.0001
Snapshots with at least 90% garbled rows: 3
count    14.000
mean      0.604
std       0.260
min       0.250
25%       0.375
50%       0.562
75%       0.729
max       1.000
Name: garbled, dtype: float64

'Ð˜ÐŸ Ð“Ñ€Ð¸Ð½ÑŒÐºÐ¾' -> 'ИП Гринько'
'å±±ä¸œæ³°ä¸°é›†å›¢' -> '山东泰丰集团'
'SÄ…d Rejonowy dla Krakowa-Krowodrzy' -> 'Sąd Rejonowy dla Krakowa-Krowodrzy'


## 26. Summary of what we learned

**Confirmed**
* About 95% of jobs carry over to the next snapshot. Text noise (case, spaces, punctuation) explains a large share of the misses.
* Renames can be recovered with the LinkedIn URL (46% of rows have one) or the start date (about a third).
* More than half of the jobs with no match come back later, so matching must use all known positions, including carried forward ones, not only the previous snapshot.
* Only 4.5% of new looking rows are truly new jobs.
* 20.9% of jobs share a company with another job of the same person, so company only matching is risky.

**Not yet decided**
* Whether to use start date alone as evidence of a rename (with a title guard).
* Whether to keep company only matching as a last resort.
* What the assignment means by a corrupted snapshot.

## 27. How many snapshots would each corruption rule discard?

In [25]:
f = df[['user_id', 'observed_ts', 'startdate', 'enddate', 'n_filled']].copy()
flags = pd.DataFrame(index=f.index)
flags['end before start'] = f['startdate'].notna() & f['enddate'].notna() & (f['enddate'] < f['startdate'])
flags['start before 1950'] = f['startdate'] < pd.Timestamp('1950-01-01')
flags['start after snapshot'] = f['startdate'] > f['observed_ts']
flags['end over 5 years after snapshot'] = f['enddate'] > f['observed_ts'] + pd.DateOffset(years=5)
flags['row has no useful field'] = f['n_filled'] == 0

keys = [f['user_id'], f['observed_ts']]
size = f.groupby(keys).size()
out = []
for name in flags.columns:
    bad = flags[name].groupby(keys).sum()
    hit = bad[bad > 0]
    out.append({
        'rule': name,
        'bad rows': int(flags[name].sum()),
        'snapshots hit': len(hit),
        'share of snapshots': round(len(hit) / len(size), 4),
        'median share of rows bad in those snapshots': round((hit / size[hit.index]).median(), 3),
    })
display(pd.DataFrame(out))

any_bad = flags.drop(columns='row has no useful field').any(axis=1).groupby(keys).sum()
print('Snapshots hit by at least one date rule:', int((any_bad > 0).sum()))

best = f.groupby(['user_id', 'observed_ts'])['n_filled'].max()
print('Snapshots where no row has any useful field:', int((best == 0).sum()))
print('Snapshots where no row has more than 1 useful field:', int((best <= 1).sum()))

,rule,bad rows,snapshots hit,share of snapshots,median share of rows bad in those snapshots
0,end before start,88,62,0.0003,0.167
1,start before 1950,66,64,0.0003,1.000
2,start after snapshot,3,3,0.0000,0.167
3,end over 5 years after snapshot,1,1,0.0000,0.111
4,row has no useful field,78,72,0.0004,1.000


Snapshots hit by at least one date rule: 129
Snapshots where no row has any useful field: 54
Snapshots where no row has more than 1 useful field: 3727
